# 01: Data Ingestion

This notebook creates or updates the canonical **data blocks pool** by running, in sync:
- `world-grid` generation
- data harmonization
- data ingestion

It ends with a report of the **available feature schemes** and **training heads** in the pool. Continue with `01_data_preparation` to prepare data for training / evaluation sessions; its dropdowns are built from this pool.

## Environment Setup & Imports

> **Prerequisite**: Ensure this notebook is attached to a cluster with the `landseg` wheel pre-installed (e.g. Databricks compute via cluster init script or cluster libraries) or an active local virtual environment. No in-notebook pip installation is required.

In [ ]:
# standard imports
import json
import pathlib
import sys
# local imports
# Attempt importing landseg directly from site-packages (wheel install).
# Fall back to local '../src' only when running inside the repository clone.
try:
    import landseg.adapters.api as api
    import landseg.mlops as mlops
    import landseg.runtime as runtime
    import landseg.testing as testing
    import landseg.runtime.validator as validator
except ImportError:
    repo_src = pathlib.Path.cwd().parent / 'src'
    if repo_src.exists():
        sys.path.insert(0, str(repo_src))
    import landseg.adapters.api as api
    import landseg.mlops as mlops
    import landseg.runtime as runtime
    import landseg.testing as testing
    import landseg.runtime.validator as validator

## Configuration

All artifacts are written under a single **experiment root**. The ingestion output (`<root>/artifacts/ingested_data`) is the input for `01_data_preparation`; its path is printed in the pool report at the end. Settings are widgets, grouped by number:

| Widgets | Group | Notes |
| --- | --- | --- |
| `01-05` | I/O | `01` experiment root is required. `02` dummy data on generates sample inputs under `<root>/input`; off requires `03` reference extent raster and `04` raw raster manifest. `05` local scratch dir (e.g. `/local_disk0` on Databricks) stages fast ephemeral I/O. |
| `11-13` | Stage settings | Grid tiling and ingestion rebuild flag. |

Less commonly changed settings (resampling, collision policy, feature engineering) stay as code defaults in the stage cells.

In [ ]:
# set up widgets (zero-padded numbered names keep them grouped and ordered in the widget bar)
widgets = runtime.NotebookWidgets()

# 01-05 I/O: single experiment root for all artifacts + input source
exp_root = widgets.text('01_exp_root', default='../experiment', label='01. Experiment Root (output)')
use_dummy_data = widgets.boolean('02_use_dummy_data', default=True, label='02. Use Dummy Data')
reference_raster_input = widgets.text('03_reference_raster', label='03. Reference Extent Raster (if not dummy)')
raw_manifest_input = widgets.text('04_raw_manifest', label='04. Raw Raster Manifest (if not dummy)')
scratch_dir_input = widgets.text('05_scratch_dir', default='', label='05. Local Scratch Dir (e.g. /local_disk0, optional)')

# 11-13 stage settings
tile_size = widgets.int('11_tile_size', default=256, label='11. Grid | Tile Size (px)')
tile_stride = widgets.int('12_tile_stride', default=128, label='12. Grid | Tile Stride (px)')
ingest_rebuild = widgets.boolean('13_ingest_rebuild', default=False, label='13. Ingest | Force Rebuild')

In [ ]:
# 01. experiment root (required): all artifacts are written under it
if not exp_root.strip():
    raise ValueError('Widget "01. Experiment Root" is required.')
resolved_exp_root = mlops.resolve_exp_root(exp_root)
resolved_scratch_dir = scratch_dir_input.strip() or None

artifacts_root = resolved_exp_root / 'artifacts'
resolved_grid_dpath = artifacts_root / 'world_grids'
resolved_harmonize_dpath = artifacts_root / 'harmonized_data'
resolved_ingest_dpath = artifacts_root / 'ingested_data'

# 02-04. inputs: dummy data under the experiment root, or user-provided files (must exist)
if use_dummy_data:
    dummy_input_root = resolved_exp_root / 'input'
    reference_raster_fpath = dummy_input_root / 'reference_raster/sample_extent.tif'
    raw_dataset_manifest = dummy_input_root / 'raw_data/manifest.json'
else:
    dummy_input_root = None
    missing = [
        label for label, val in [
            ('03. Reference Extent Raster', reference_raster_input),
            ('04. Raw Raster Manifest', raw_manifest_input),
        ] if not val.strip()
    ]
    if missing:
        raise ValueError(f'"02. Use Dummy Data" is false; set widget(s): {", ".join(missing)}')
    reference_raster_fpath = validator.require_file(reference_raster_input.strip())
    raw_dataset_manifest = validator.require_file(raw_manifest_input.strip())

# log resolved configuration
print(f'Experiment root:          {resolved_exp_root}')
print(f"Input source:             {'dummy data' if use_dummy_data else 'user-provided'}")
print(f'Reference extent raster:  {reference_raster_fpath}')
print(f'Raw raster manifest:      {raw_dataset_manifest}')
print(f'Local scratch dir:        {resolved_scratch_dir or "(system default)"}')
print(f'Tile size / stride:       {tile_size} / {tile_stride} px')
print(f'Rebuild ingest:           {ingest_rebuild}')
print('Artifact dirs:')
for name, dpath in [
    ('world grid', resolved_grid_dpath),
    ('harmonized', resolved_harmonize_dpath),
    ('ingested', resolved_ingest_dpath),
]:
    print(f'  {name:<11} {dpath}')

In [ ]:
# for testing purposes, generate a set of real rasters (`.tif` files) under the experiment root to demo the following process
if use_dummy_data:
    assert dummy_input_root # typing
    testing.generate_pre_harmonization_data(dummy_input_root)
else:
    print('Dummy data generation skipped (user-provided I/O mode).')

## Run Pipeline Stages

Each stage reads the previous stage's artifacts under the experiment root and is configured from the widgets above. Run all three in order the first time; afterwards you can re-run a single stage after changing its widgets.

1. **World Grid**: canonical spatial reference frame (CRS, resolution, tile size, stride, extent) used by all later stages and inference.
2. **Harmonization**: reprojects and stacks raw rasters into Virtual Rasters (`.vrt`) aligned to the world grid.
3. **Ingestion**: slices harmonized rasters into canonical data blocks (`.npz`) and applies the collision policy against the block pool. (`BatchIngestConfigurator` can catch up all pending batches in `harmonization_runs.json`.)

In [ ]:
# 1. world grid
grid_configurator = api.WorldGridConfigurator(output_dpath=resolved_grid_dpath)
if resolved_scratch_dir:
    grid_configurator.set_scratch(resolved_scratch_dir)

# set world grid specs
grid_configurator.set_grid(
    tile_size=tile_size,
    tile_stride=tile_stride,
    reference_raster=reference_raster_fpath,
)

# run world-grid pipeline
api.run(grid_configurator.running_root_config)

In [ ]:
# 2. data harmonization
harmonize_configurator = api.DataHarmonizationConfigurator(
    upstream_dpath=resolved_grid_dpath,
    output_dpath=resolved_harmonize_dpath,
)
if resolved_scratch_dir:
    harmonize_configurator.set_scratch(resolved_scratch_dir)

# set dataset manifest containing metadata for all input rasters
harmonize_configurator.set_dataset_manifest(
    dataset_manifest=raw_dataset_manifest
)

# set resampling algorithms for continuous and categorical rasters
harmonize_configurator.set_resampling(
    continuous='bilinear',
    categorical='nearest'
)

# set target spatial world grid tile dimensions (matches world grid generated above)
harmonize_configurator.set_grid(
    tile_size=tile_size,
    tile_stride=tile_stride
)

# run data-harmonize pipeline
api.run(harmonize_configurator.running_root_config)

In [ ]:
# 3. data ingestion
ingest_configurator = api.DataIngestionConfigurator(
    upstream_dpath=resolved_harmonize_dpath,
    output_dpath=resolved_ingest_dpath,
)
if resolved_scratch_dir:
    ingest_configurator.set_scratch(resolved_scratch_dir)

# force rebuild ingestion data artifacts (widget 13)
ingest_configurator.set_rebuild(ingest_rebuild)

# set collision policy ('skip', 'overwrite', or 'error')
ingest_configurator.set_collision_policy('skip')

# optionally configure feature engineering (e.g. slope, ndvi)
ingest_configurator.set_feature_engineering(
    add_topo=['slope', 'tpi'],
    add_spectral=['ndvi', 'ndmi', 'nbr']
)

# run batch ingestion workflow
api.run(ingest_configurator.running_root_config)

## Data Blocks Pool Report

Summarizes the pool's `data_blocks/schema.json`: the **feature schemes** (named band selections per feature dataset) and the **training heads** (label datasets and their target schemes). `01_data_preparation` builds its dropdowns from this same file.

In [ ]:
pool_dpath = resolved_ingest_dpath / 'data_blocks'
pool_schema = json.loads(validator.require_file(pool_dpath / 'schema.json').read_text(encoding='utf-8'))
catalog = json.loads((pool_dpath / 'catalog.json').read_text(encoding='utf-8'))

dataset = pool_schema['dataset']
image_band_map = pool_schema['io_conventions']['image_band_map']
label_band_map = pool_schema['io_conventions']['label_band_map']
labels = pool_schema['labels']

print(f'Ingestion root (use as 01_data_preparation widget "01. Ingestion Root"): {resolved_ingest_dpath}')
print(f'Blocks: {len(catalog)} | grids: {", ".join(dataset["mapped_grids"])} | last updated: {dataset["last_updated"]}')

print(f'\nAvailable feature schemes (default "all" = {len(image_band_map)} bands: {", ".join(image_band_map)})')
for ds_name, schemes in dataset.get('image_schemes', {}).items():
    for scheme, bands in schemes.items():
        print(f'  {ds_name:<12} {scheme:<10} {", ".join(bands)}')

print('\nAvailable training heads (target schemes; "raw" = canonical classes)')
for head in label_band_map:
    n_cls = labels['label_num_cls'].get(head)
    class_names = ', '.join(labels.get('label_class_names', {}).get(head, []))
    print(f'  {head:<12} raw ({n_cls} classes: {class_names})')
    for scheme, spec in dataset.get('label_schemes', {}).get(head, {}).items():
        reclass_names = ', '.join(spec.get('reclass_name', {}).values())
        print(f'  {"":<12} {scheme} ({len(spec.get("reclass", {}))} classes: {reclass_names})')